# 05 — Validation temporelle, data leakage et drift

Découper le dataset transformé, auditer les fuites possibles et mesurer les différences de distribution.
**Dataset principal uniquement ; aucun modèle entraîné.** Kernel : **Python (Telco Churn)**.


## 1. Choisir le scénario de validation

**Question.** Comment évaluer une prédiction sur de futurs snapshots, sans mélanger passé et futur ?

| Partition | Fenêtre configurée | Usage |
|---|---|---|
| Train | Janvier–juin 2024 | Futurs apprentissages et backtests internes. |
| Validation | Août–septembre 2024 | Choix de modèles et réglages ultérieurs. |
| Test de développement | Novembre–décembre 2024 | Comparaison finale de développement, pas test entièrement intact. |

Juillet et octobre restent hors des fenêtres. Une purge retire aussi toute ligne dont la fenêtre supposée
de label et son délai de disponibilité atteindraient la partition suivante. **L'horizon de 30 jours est une
hypothèse non confirmée**, paramétrée dans `configs/temporal_validation.yaml`.

La stratégie principale autorise les clients communs entre périodes : elle vise la continuation du panel,
sans utiliser l'identifiant comme feature. Les snapshots ne doivent jamais être communs. Une variante sans
clients partagés sera examinée séparément, car elle change la population et la question d'évaluation.

Le fichier 2024 a déjà servi à l'EDA : le test de cette année ne doit pas être présenté comme une évaluation
finale aveugle. Une période nouvelle restera nécessaire pour cette prétention.


In [11]:
import importlib

for module_name in (
    "telco_churn.settings",
    "telco_churn.modeling.temporal_split",
    "telco_churn.monitoring.data_drift",
    "telco_churn.visualization.validation",
):
    _ = importlib.reload(importlib.import_module(module_name))

In [12]:
import pandas as pd
from IPython.display import display

from telco_churn.logging import configure_logging
from telco_churn.modeling.temporal_split import (
    TemporalConfig,
    TemporalSplitter,
    load_processed,
    save_temporal_partitions,
)
from telco_churn.monitoring.data_drift import DataDriftMonitor
from telco_churn.settings import Settings
from telco_churn.visualization.validation import (
    plot_drift_summary,
    plot_partition_target_balance,
    plot_reference_distribution,
    plot_split_timeline,
)

settings = Settings.load()
configure_logging(settings)
config = TemporalConfig.load(settings)
df, source_manifest = load_processed(settings)
feature_columns = source_manifest["feature_columns"]
target = settings.churn_target_column
splitter = TemporalSplitter(config, target)
print(f"Dataset transformé : {len(df):,} lignes | {len(feature_columns)} features candidates")

Dataset transformé : 38,689 lignes | 17 features candidates


## 2. Construire les partitions et tracer les exclusions

**Question.** Quelles lignes sont conservées, et pourquoi les autres sont-elles écartées ?

Le split retire les copies de snapshots équivalentes dans le dataset transformé. Une même clé client/date
avec des valeurs contradictoires déclenche une erreur, plutôt qu'un choix arbitraire de la première ligne.
Les cibles manquantes sont exclues ; dates/identifiants invalides empêchent le découpage.
Chaque ligne se retrouve dans une partition ou dans le journal des exclusions.


In [13]:
result = splitter.split(df)
split_summary = result.summary(config, target)
display(split_summary)
print("Lignes exclues, par motif")
display(result.excluded["exclusion_reason"].value_counts().rename("n_rows").to_frame())
fig = plot_split_timeline(split_summary, config)
fig.show(renderer="plotly_mimetype")

,partition,n_rows,n_clients,first_date,last_date,n_positive_snapshots,n_positive_clients,positive_percent
0,train,20103,3762,2024-01-31,2024-06-30,590,590,2.934885
1,validation,6309,3239,2024-08-31,2024-09-30,209,209,3.312728
2,test,5910,3003,2024-11-30,2024-12-31,165,165,2.791878


Lignes exclues, par motif


,n_rows
exclusion_reason,
outside_windows,6307
duplicate_equivalent_snapshot,60


### Analyse des résultats

Sur les données actuelles : **20 103 lignes de train, 6 309 de validation et 5 910 de test**.
Les **60 copies équivalentes** et **6 307 lignes de juillet/octobre** sont journalisées séparément.
La purge ne retire pas de ligne supplémentaire avec l'horizon supposé de 30 jours et ces fenêtres espacées ;
changer cet horizon peut changer les effectifs.



## 3. Quelle est la répartition du churn dans chaque partition ?

Chaque partition comporte deux barres côte à côte : **bleu = non-churn**, **rouge = churn**.
Leur hauteur indique la proportion de chaque classe, avec le pourcentage au-dessus de chaque barre.
Le survol donne les effectifs de chaque classe et le total. Les proportions sont calculées après les exclusions du split.


In [14]:
fig = plot_partition_target_balance(split_summary)
fig.show(renderer="plotly_mimetype")

### Analyse des résultats

| Partition | Churn | Non-churn | Snapshots positifs / total |
|---|---:|---:|---:|
| Train | 2,93 % | 97,07 % | 590 / 20 103 |
| Validation | 3,31 % | 96,69 % | 209 / 6 309 |
| Test | 2,79 % | 97,21 % | 165 / 5 910 |

La classe positive reste rare dans les trois partitions. La validation a une proportion légèrement supérieure
au train et au test ; le découpage temporel conserve cette variation au lieu de rééquilibrer artificiellement
les classes. Ces proportions concernent des snapshots, pas des clients indépendants.


## 4. Auditer le data leakage sur les partitions réelles

**Question.** Qu'est-ce qui est vérifié, et qu'est-ce qui reste impossible à certifier avec les informations actuelles ?

- **PASS** : contrôle technique satisfait, sous les paramètres déclarés.
- **FAIL** : violation technique ; export refusé.
- **INFO** : chevauchement attendu par la stratégie choisie.
- **WARN** : limite connue à prendre en compte.
- **UNVERIFIED** : information métier ou preuve manquante.

Les tests unitaires vérifient le code ; ce rapport vérifie les partitions effectivement obtenues.
Il contrôle ordre temporel, fin des fenêtres supposées de label, clés communes, features interdites et signaux
numériques exactement égaux à la cible sur le train. Une égalité suspecte n'est pas à elle seule une preuve de fuite.


In [15]:
leakage_report = splitter.leakage_report(result, source_manifest)
display(leakage_report)

,check,status,detail
0,feature_exclusions,PASS,Colonnes interdites dans X : []
1,snapshot_overlap_train_validation,PASS,0 clés communes
2,client_overlap_train_validation,INFO,3062 clients communs ; politique recurring
3,snapshot_overlap_train_test,PASS,0 clés communes
4,client_overlap_train_test,INFO,2812 clients communs ; politique recurring
5,snapshot_overlap_validation_test,PASS,0 clés communes
6,client_overlap_validation_test,INFO,2950 clients communs ; politique recurring
7,chronology_and_purge_train_validation,PASS,"Purge de 30 jours, selon horizon déclaré"
8,chronology_and_purge_validation_test,PASS,"Purge de 30 jours, selon horizon déclaré"
9,label_horizon,UNVERIFIED,30 jours : hypothèse


### Analyse des résultats

Aucun snapshot client/date n'est partagé entre partitions et la chronologie purgée est respectée sous
l'hypothèse déclarée. **3 062 clients** sont communs au train/validation et **2 812** au train/test : c'est
autorisé dans le scénario principal, avec identifiant exclu de la matrice et features connues au snapshot à confirmer.

**On ne peut pas déclarer le dataset exempt de toute fuite.** L'horizon réel, la disponibilité des 17 features
et la maturité des labels de test ne sont pas confirmés. La période de test a déjà été explorée dans l'EDA.
Ces limites restent visibles dans le manifeste ; un bon ordre des dates ne suffit pas à les résoudre.


## 5. Comparer une stratégie sans clients communs

**Question.** L'évaluation serait-elle la même si les partitions portaient sur des clients distincts ?

Le mode `disjoint` conserve les mêmes dates, mais écarte des partitions suivantes les clients déjà
présents dans une partition précédente. Il évalue des clients nouveaux pour l'échantillon d'entraînement,
pas nécessairement nouveaux pour toute l'entreprise. Les exclusions changent donc la population évaluée.


In [16]:
disjoint_config = TemporalConfig.model_validate(
    {**config.model_dump(), "customer_policy": "disjoint"}
)
disjoint_result = TemporalSplitter(disjoint_config, target).split(df, require_nonempty=False)
display(disjoint_result.summary(disjoint_config, target))

,partition,n_rows,n_clients,first_date,last_date,n_positive_snapshots,n_positive_clients,positive_percent
0,train,20103,3762,2024-01-31,2024-06-30,590,590,2.934885
1,validation,281,177,2024-08-31,2024-09-30,30,30,10.676157
2,test,101,53,2024-11-30,2024-12-31,10,10,9.900990


### Analyse des résultats

La variante ne conserve que **281 lignes / 177 clients en validation** et **101 lignes / 53 clients en test**,
avec respectivement **30 et 10 positifs**. Les taux passent à environ **10,68 % et 9,90 %** : la population a
fortement changé. Cette variante ne doit pas être choisie uniquement pour éliminer les identifiants communs.
Elle est pertinente si la question métier vise ces clients non vus, mais son test est ici très limité.


## 6. Préparer les backtests à fenêtres croissantes

**Question.** Comment vérifier la stabilité d'un futur modèle sans réutiliser le test de développement pour les réglages ?

Les deux folds configurés restent entièrement dans le **train principal**. Chaque fold réappliquera la purge ;
les futurs prétraitements appris devront être ajustés sur son train seulement. Aucun modèle ni imputation
n'est appris dans ce notebook.


In [17]:
backtests = splitter.backtests(result)
backtest_summary = pd.DataFrame(
    [
        {
            "fold": name,
            "partition": partition,
            "n_rows": len(frame),
            "n_clients": frame[config.id_column].nunique(),
            "first_date": frame[config.date_column].min(),
            "last_date": frame[config.date_column].max(),
            "n_positive_snapshots": int(frame[target].sum()),
        }
        for name, partitions in backtests.items()
        for partition, frame in partitions.items()
    ]
)
display(backtest_summary)

,fold,partition,n_rows,n_clients,first_date,last_date,n_positive_snapshots
0,fold_1,train,6910,3533,2024-01-31,2024-02-29,219
1,fold_1,validation,3299,3299,2024-04-30,2024-04-30,85
2,fold_2,train,13569,3624,2024-01-31,2024-04-30,410
3,fold_2,validation,3250,3250,2024-06-30,2024-06-30,78


### Analyse de la stratégie

Les folds utilisent février → avril et avril → juin comme fins de train et mois d'évaluation. Ils permettent
une comparaison prospective à l'intérieur de la période d'apprentissage. Leur historique reste court : une
variabilité de scores future devra être documentée, et ces folds ne rendent pas le test 2024 à nouveau aveugle.


## 7. Mesurer le drift par rapport au train

**Question.** Les distributions de validation et de test diffèrent-elles de la référence d'apprentissage ?

Le moniteur apprend les bornes numériques et les niveaux de référence **sur le train uniquement**, puis
compare validation et test sans refit. Les manquants ont leur propre modalité ; les variables numériques
à faible cardinalité sont aussi surveillées comme modalités, pour ne pas masquer un changement de constante.

Les indicateurs sont le **PSI**, la **variation totale**, les écarts de manquants et les nouvelles modalités.
Les proportions hors plage numérique du train sont également fournies. Les seuils YAML sont heuristiques,
pas des p-values ; aucun test d'indépendance des snapshots répétés n'est supposé.

**NO_ALERT signifie absence d'alerte selon ces métriques, pas preuve d'absence de drift.** Le drift conditionnel
ou multivarié et la dérive de performance d'un futur modèle ne sont pas certifiés par ce diagnostic univarié.


In [18]:
drift_monitor = DataDriftMonitor(config.drift).fit(result.partitions["train"][feature_columns])
drift_report = pd.concat(
    [
        drift_monitor.compare(result.partitions[name][feature_columns]).assign(
            comparison=f"train → {name}"
        )
        for name in ["validation", "test"]
    ],
    ignore_index=True,
)
fig = plot_drift_summary(drift_report, config.drift)
fig.show(renderer="plotly_mimetype")
print("Alertes détectées")
display(drift_report.loc[drift_report["severity"].ne("NO_ALERT")])

Alertes détectées


,feature,kind,psi,total_variation,train_missing_percent,other_missing_percent,missing_delta_pp,unseen_percent_of_nonmissing,outside_train_range_percent,reference_status,severity,n_train,n_other,comparison
17,tenure_months,numeric,0.129092,0.126461,0.417848,0.35533,-0.062518,0.0,7.199864,available,WARNING,20103,5910,train → test
18,log1p_tenure_months,numeric,0.129092,0.126461,0.417848,0.35533,-0.062518,0.0,7.199864,available,WARNING,20103,5910,train → test


### Analyse des résultats

Aucune alerte n'est déclenchée en validation avec les seuils actuels. Dans le test, **`tenure_months` et son
logarithme ont un PSI d'environ 0,129**, au-dessus du seuil d'avertissement 0,10, mais sous le seuil élevé 0,25.
Ce sont deux représentations du même signal, pas deux preuves indépendantes de dérive.

Environ **7,20 % des anciennetés renseignées du test** dépassent la plage du train. Le vieillissement du panel
peut y contribuer ; il n'est pas nécessairement une erreur. Le taux de NPS manquant passe d'environ 67,44 %
dans le train à 68,54 % dans le test. Les écarts de prévalence de cible restent distincts du drift des features.
On ne déplace pas les dates pour faire disparaître ces écarts ; il faudra examiner leur effet sur les futurs scores.


## 8. Comprendre l'alerte sur l'ancienneté

**Question.** Où la distribution change-t-elle lorsque les mêmes intervalles de référence sont utilisés ?
Les bornes ci-dessous viennent du train ; les autres périodes ne redéfinissent pas les classes.


In [19]:
tenure_distribution = drift_monitor.distribution_table(
    "tenure_months",
    {name: result.partitions[name][feature_columns] for name in ["validation", "test"]},
)
fig = plot_reference_distribution(tenure_distribution, "tenure_months")
fig.show(renderer="plotly_mimetype")

### Analyse de la distribution

Les proportions par intervalle montrent le déplacement du panel vers certaines durées. Le dernier intervalle
reste ouvert : une masse hors de la plage historique peut y être regroupée, d'où l'intérêt du compteur hors plage
en complément du PSI. Ces parts sont pondérées par les snapshots, pas par des clients indépendants.


## 9. Sauvegarder les partitions et leurs diagnostics

Les fichiers de développement vont dans `data/processed/temporal_splits/`. Ils conservent `source_row`
pour la traçabilité ; ce marqueur, l'identifiant, la date et la cible ne sont pas des features du modèle.
Le manifeste distingue ces colonnes des 17 features et reste **`training_validated: false`**.
Un contrôle technique `FAIL` empêche l'export ; les confirmations manquantes restent explicitement documentées.


In [20]:
split_path = save_temporal_partitions(
    result, source_manifest, config, settings.split_output_dir, leakage_report, drift_report
)
print(f"Partitions et rapports sauvegardés : {split_path}")

Partitions et rapports sauvegardés : /Users/surelmanda/2-LLops-Databricks-Projects/Telco-Churn-Optimization/data/processed/temporal_splits


## Synthèse

- **Stratégie :** dates fixées, intervalles non utilisés et purge des fenêtres de label supposées.
- **Intégrité :** snapshots équivalents dédupliqués, contradictions refusées et exclusions tracées.
- **Clients :** chevauchements mesurés et assumés pour la continuation du panel ; variante disjointe distincte.
- **Drift :** alerte d'ancienneté dans le test ; aucune garantie d'absence de drift ou de fuite métier.
- **À confirmer :** horizon, disponibilité des sources, maturité des labels et nouvelle période d'évaluation non explorée.

Le notebook 06 pourra construire une baseline de développement avec apprentissage des prétraitements sur
train uniquement, sous ces limites documentées. Ce notebook ne certifie pas un score final exploitable en production.
